# 1. Select papers

Start of the **altendor** pipeline. This notebook:

1. Runs an OpenAlex keyword (or topic / concept) search to gather candidate DOIs in the science-of-science space.
2. Joins those DOIs to Altmetric on BigQuery to pick up the Altmetric attention score and `ro_id` (Altmetric output id) for each paper.
3. Picks the top-N papers by score, with optional **pinned DOIs** that always get through.
4. Enriches with abstracts from OpenAlex (the GBQ `research_outputs` table has no abstract column).
5. Writes `output/<run_id>/papers.parquet` plus a `manifest.json`.

Each BigQuery step has a separate **dry-run cell** and **real-run cell** — inspect bytes-to-scan before running.

See `PIPELINE_PLAN.md` § S18 / S3 / S4 / S5 for the contract.

## Setup

Set the run id, force ADC for BigQuery, and import the helpers.

In [1]:
import os
from datetime import datetime, timezone
from pathlib import Path

# Force ADC over any GOOGLE_APPLICATION_CREDENTIALS that may be set;
# Altmetric's Analytics Hub uses row-level security keyed to your user.
os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

RUN_ID = datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H-%M")
OUT_DIR = OUTPUT_ROOT / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

Run id: 2026-06-28T21-34
Output dir: /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34


In [2]:
from google.cloud import bigquery
import httpx

bq_client = bigquery.Client()
openalex_client = httpx.Client(timeout=30.0)
print(f"BigQuery project: {bq_client.project}")

BigQuery project: altmetric-endorsements


## OpenAlex search

Default: free-text keyword `science of science`. Override by setting `TOPIC_ID` or `CONCEPT_ID` and re-running.

**Pinned DOIs**: edit `PINNED_DOIS` to force-include specific seed papers (e.g. famous metascience pieces) regardless of their score.

Tip: `resolve_topic_by_name("science of science")` shows OpenAlex's top-matching topic id if you want to switch to a stricter topic filter.

In [3]:
from altendor.sources.openalex import resolve_topic_by_name, search_dois

QUERY = "science of science"
TOPIC_ID: str | None = None      # e.g. "T10058" for Scientometrics
CONCEPT_ID: str | None = None    # e.g. "C520712124"
N_FROM_OPENALEX = 200            # cap; we'll trim again after Altmetric filtering

PINNED_DOIS: list[str] = [
    # "10.1126/science.aap8731",   # Funk & Owen-Smith — disruption index
    # "10.1371/journal.pmed.0020124",  # Ioannidis — Why Most Published Research Findings Are False
]

topic_hint = resolve_topic_by_name(QUERY, client=openalex_client) if (TOPIC_ID is None and CONCEPT_ID is None) else None
if topic_hint:
    print(f"Top OpenAlex topic matching {QUERY!r}: {topic_hint['id']} \u2014 {topic_hint['display_name']} ({topic_hint['works_count']} works). Set TOPIC_ID above to use it.")

dois = search_dois(QUERY, topic_id=TOPIC_ID, concept_id=CONCEPT_ID, n=N_FROM_OPENALEX, client=openalex_client)
print(f"OpenAlex returned {len(dois)} DOIs.")
dois[:10]

Top OpenAlex topic matching 'science of science': T10192 — Catalytic Processes in Materials Science (224150 works). Set TOPIC_ID above to use it.
OpenAlex returned 200 DOIs.


['10.2307/1270020',
 '10.1016/0198-9715(90)90050-4',
 '10.2307/3498751',
 '10.1108/ir.1999.04926fae.001',
 '10.3758/bf03193146',
 '10.2307/2290095',
 '10.7312/pric91844',
 '10.4324/9780203771587',
 '10.4324/9781410606266',
 '10.1111/(issn)1540-6237']

## Join DOIs to Altmetric attention

We look up each DOI in Altmetric's `research_outputs` table. DOIs not present in Altmetric simply drop out.

**Dry-run cell first**, then the real-run cell.

In [4]:
from altendor.bigquery.preflight import preflight
from altendor.bigquery.queries import research_outputs_by_dois

candidate_dois = sorted(set(dois) | set(PINNED_DOIS))
print(f"{len(candidate_dois)} unique candidate DOIs (incl. {len(PINNED_DOIS)} pinned)")

ro_sql, ro_cfg = research_outputs_by_dois(candidate_dois)
preflight(bq_client, ro_sql, job_config=ro_cfg)

200 unique candidate DOIs (incl. 0 pinned)
Preflight: dry-run bytes masked by row-level security; upper bound from referenced tables:
  - altmetric_on_gbq.research_outputs: 18431.91 MB  (53,898,448 rows)
Upper-bound scan: 18431.91 MB


18431909882

In [5]:
from altendor.sources.altmetric import join_dois_to_attention

papers_df = join_dois_to_attention(bq_client, candidate_dois)
print(f"{len(papers_df)} papers matched in Altmetric (of {len(candidate_dois)} candidates).")
papers_df.sort_values("altmetric_score", ascending=False).head(10)

111 papers matched in Altmetric (of 200 candidates).


,ro_id,doi,title,altmetric_score,last_mentioned_at
17,4443094,10.1126/science.aac4716,Estimating the reproducibility of psychologica...,4391.35,2026-06-23 11:09:31+00:00
41,80902343,10.1038/s41562-020-0884-z,Using social and behavioural science to suppor...,2999.20,2026-01-26 08:43:07+00:00
59,2345524,10.1073/pnas.1319030111,Active learning increases student performance ...,2605.60,2026-06-09 01:15:03+00:00
37,15362282,10.1038/s41562-016-0021,A manifesto for reproducible science,2210.75,2026-06-13 22:36:20+00:00
42,79872111,10.1016/s2215-0366(20)30168-1,Multidisciplinary research priorities for the ...,2195.15,2025-08-04 00:21:06+00:00
25,34106210,10.1126/science.aao2998,The science of fake news,2068.25,2026-06-23 20:29:07+00:00
35,150735611,10.1017/9781009157896,Climate Change 2021 – The Physical Science Basis,1021.10,2026-06-27 23:33:23+00:00
24,33825242,10.1126/science.aao0185,Science of science,683.35,2026-06-09 00:00:00+00:00
72,4252201,10.1017/cbo9781107415324,Climate change 2013,658.90,2026-06-15 15:03:58+00:00
4,21599659,10.1126/science.159.3810.56,The Matthew Effect in Science,634.30,2026-06-21 09:20:16+00:00


## Pick top-N papers (pinned always included)

`top_papers` drops zero-score rows from the ranked pool, takes the top-K by `altmetric_score`, and ensures any pinned DOI is included (and appears first).

In [6]:
from altendor.sources.altmetric import top_papers

K = 10
selected = top_papers(papers_df, k=K, pinned_dois=PINNED_DOIS)
print(f"Selected {len(selected)} papers (k={K} ranked + pinned).")
selected[["doi", "ro_id", "altmetric_score", "title"]]

Selected 10 papers (k=10 ranked + pinned).


,doi,ro_id,altmetric_score,title
0,10.1126/science.aac4716,4443094,4391.35,Estimating the reproducibility of psychologica...
1,10.1038/s41562-020-0884-z,80902343,2999.20,Using social and behavioural science to suppor...
2,10.1073/pnas.1319030111,2345524,2605.60,Active learning increases student performance ...
3,10.1038/s41562-016-0021,15362282,2210.75,A manifesto for reproducible science
4,10.1016/s2215-0366(20)30168-1,79872111,2195.15,Multidisciplinary research priorities for the ...
5,10.1126/science.aao2998,34106210,2068.25,The science of fake news
6,10.1017/9781009157896,150735611,1021.10,Climate Change 2021 – The Physical Science Basis
7,10.1126/science.aao0185,33825242,683.35,Science of science
8,10.1017/cbo9781107415324,4252201,658.90,Climate change 2013
9,10.1126/science.159.3810.56,21599659,634.30,The Matthew Effect in Science


## Enrich abstracts from OpenAlex

Altmetric's `research_outputs` table doesn't carry abstracts. We call OpenAlex per-DOI to fetch `abstract_inverted_index` and reconstruct the abstract. Failures (missing DOI in OpenAlex, network errors) leave `abstract` as `None`.

In [7]:
from altendor.sources.altmetric import enrich_with_openalex_abstracts

enriched = enrich_with_openalex_abstracts(selected, http_client=openalex_client)
n_with_abs = enriched["abstract"].notna().sum()
print(f"{n_with_abs}/{len(enriched)} papers have abstracts after enrichment.")
enriched[["doi", "title", "abstract"]].head(3)

8/10 papers have abstracts after enrichment.


,doi,title,abstract
0,10.1126/science.aac4716,Estimating the reproducibility of psychologica...,Reproducibility is a defining feature of scien...
1,10.1038/s41562-020-0884-z,Using social and behavioural science to suppor...,NaN
2,10.1073/pnas.1319030111,Active learning increases student performance ...,To test the hypothesis that lecturing maximize...


## Write outputs

Persist `papers.parquet` for downstream notebooks, plus a small `manifest.json` for traceability. The `output/<run_id>/` directory is gitignored except for manifests.

In [8]:
import json

papers_path = OUT_DIR / "papers.parquet"
enriched.to_parquet(papers_path, index=False)

manifest = {
    "run_id": RUN_ID,
    "stage": "1_select_papers",
    "openalex_query": QUERY,
    "openalex_topic_id": TOPIC_ID,
    "openalex_concept_id": CONCEPT_ID,
    "openalex_n_returned": len(dois),
    "pinned_dois": list(PINNED_DOIS),
    "altmetric_matched": int(len(papers_df)),
    "selected_k": int(len(enriched)),
    "with_abstract": int(n_with_abs),
    "output_files": [str(papers_path.relative_to(REPO_ROOT))],
}
manifest_path = OUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {papers_path}\nWrote {manifest_path}")

Wrote /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34/papers.parquet
Wrote /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34/manifest.json
